# 마스터 테이블 구축 및 스케일링

> **이 노트북은 `C123_master_table.csv` 및 `C123_scaled_master_table.csv` 생성 과정을 문서화합니다.
> 제출 최종 결과물은 `csv/processed/` 폴더의 CSV 파일입니다.**

**파이프라인 개요**

| 단계 | 내용 | 출력 |
|------|------|------|
| PART 1 (STEP 1~7) | 원천 데이터 수집·결합·결측치 처리 | `C123_master_table.csv` |
| PART 2 (STEP 8~10) | C1/C2/C3 스케일링 + CRITIC 가중치 산출 | `C123_scaled_master_table.csv` |

**입력 파일 위치**: `../csv/raw/`, `../csv/processed/`  
**출력 파일 위치**: `../csv/processed/`

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import RobustScaler, QuantileTransformer

RAW_DIR = "../csv/raw"
OUT_DIR = "../csv/processed"
os.makedirs(OUT_DIR, exist_ok=True)

---
## PART 1. 마스터 테이블 구축

### STEP 1. 한국 → EU 수출 집계

In [ ]:
print("=" * 60)
print("STEP 1. Korea EU 수출 집계")
print("=" * 60)

eu_files = sorted(glob.glob(f"{RAW_DIR}/raw_customs_eu_*.csv"))
print(f"  파일 {len(eu_files)}개 발견")

eu_dfs = []
for f in eu_files:
    eu_dfs.append(pd.read_csv(f, dtype={"hs6": str}))
eu_raw = pd.concat(eu_dfs, ignore_index=True)
eu_raw["hs6"] = eu_raw["hs6"].astype(str).str.zfill(6)

step1 = (
    eu_raw
    .groupby(["hs6", "year"], as_index=False)
    .agg(
        kr_eu_export_weight_kg = ("export_weight_kg", "sum"),
        kr_eu_export_value_usd = ("export_value_usd", "sum"),
    )
)
print(f"  ✅ {len(step1):,}행\n")

### STEP 2. 한국 → 전세계 수출 집계

In [ ]:
print("=" * 60)
print("STEP 2. Korea 전세계 수출")
print("=" * 60)

world_files = sorted(glob.glob(f"{RAW_DIR}/raw_customs_world_*.csv"))
print(f"  파일 {len(world_files)}개 발견")

world_dfs = []
for f in world_files:
    world_dfs.append(pd.read_csv(f, dtype={"hs6": str}))
world_raw = pd.concat(world_dfs, ignore_index=True)
world_raw["hs6"] = world_raw["hs6"].astype(str).str.zfill(6)

step2 = (
    world_raw
    .groupby(["hs6", "year"], as_index=False)
    .agg(
        kr_world_export_weight_kg = ("export_weight_kg", "sum"),
        kr_world_export_value_usd = ("export_value_usd", "sum"),
    )
)
print(f"  ✅ {len(step2):,}행\n")

### STEP 3. 탄소집약도(ci) 파싱

EU 집행위 기본값 파일(IR 2025/2621, `DVs as adopted_v20260204.xlsx`)에서 한국·중국·일본·인도·튀르키예·미국의 HS6별 탄소집약도를 추출합니다.

In [ ]:
print("=" * 60)
print("STEP 3. 탄소집약도 파싱")
print("=" * 60)

TARGET_SHEETS = {
    "South Korea":   "kor",
    "China":         "chn",
    "Japan":         "jpn",
    "India":         "ind",
    "Türkiye":       "tur",
    "United States": "usa",
}

xlsx_path = f"{RAW_DIR}/DVs as adopted_v20260204.xlsx"
xl = pd.ExcelFile(xlsx_path)
ci_dict = {}

for sheet_name, iso in TARGET_SHEETS.items():
    if sheet_name not in xl.sheet_names:
        print(f"  ⚠️  {sheet_name} 시트 없음")
        continue

    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    df.columns = df.iloc[1]
    df = df.iloc[2:].reset_index(drop=True)
    df.columns = [str(c).strip().replace("\n", " ") for c in df.columns]

    cn_col = df.columns[0]
    total_col = [c for c in df.columns if "total emissions" in c.lower() and "mark-up" not in c.lower()]
    if not total_col:
        print(f"  ⚠️  {sheet_name}: total emissions 컬럼 없음")
        continue
    total_col = total_col[0]

    rows = []
    for _, row in df.iterrows():
        cn_code = str(row[cn_col]).strip().replace(" ", "")
        val = row[total_col]

        if cn_code in ["nan", "ProductCNCode", "Cement", "Steel",
                       "Aluminium", "Fertilisers", "Hydrogen"]:
            continue
        if not cn_code[0].isdigit():
            continue
        if str(val).strip() in ["–", "_", "-", "nan", "N.A."]:
            val = np.nan
        else:
            try:
                val = float(val)
            except:
                val = np.nan

        hs6 = cn_code.replace(" ", "").replace(".", "")[:6].zfill(6)
        rows.append({"hs6": hs6, f"ci_{iso}": val})

    ci_df = pd.DataFrame(rows)
    ci_df = (
        ci_df.dropna(subset=[f"ci_{iso}"])
             .groupby("hs6")[f"ci_{iso}"]
             .mean()
             .reset_index()
    )
    ci_dict[iso] = ci_df
    print(f"  {sheet_name}: {len(ci_df)}개")

step3 = None
for iso, ci_df in ci_dict.items():
    if step3 is None:
        step3 = ci_df
    else:
        step3 = step3.merge(ci_df, on="hs6", how="outer")

print(f"  ✅ {len(step3):,}행\n")

### STEP 4. ETS 가격 (EUA / KAU 연평균)

In [ ]:
print("=" * 60)
print("STEP 4. ETS 가격")
print("=" * 60)

eua = pd.read_csv(f"{OUT_DIR}/EUA_다통화_연평균가격.csv")
kau = pd.read_csv(f"{OUT_DIR}/KAU_다통화_연평균가격.csv")

eua = eua.rename(columns={"Year": "year", "VWAP_EUR": "eua_price_eur"})
kau_year_col = kau.columns[0]
kau = kau.rename(columns={kau_year_col: "year", "VWAP_EUR": "kau_price_eur"})

step4 = eua[["year", "eua_price_eur"]].merge(
    kau[["year", "kau_price_eur"]], on="year", how="outer"
)
print(f"  ✅ {len(step4)}행\n")

### STEP 5. 경쟁국 EU 수출량

중국·인도·일본·튀르키예·미국의 EU向 수출중량을 HS6·연도별로 집계합니다.  
`net_weight_kg` 결측이고 `qty_unit=8`(중량 단위)인 행은 `qty` 값으로 대체합니다.

In [ ]:
print("=" * 60)
print("STEP 5. 경쟁국 EU 수출량")
print("=" * 60)

df_b = pd.read_csv(f"{RAW_DIR}/table_B_final.csv", dtype={"hs6": str})
df_b["hs6"] = df_b["hs6"].astype(str).str.zfill(6)

mask_kg = (df_b["net_weight_kg"].isna()) & (df_b["qty_unit"] == 8)
df_b.loc[mask_kg, "net_weight_kg"] = df_b.loc[mask_kg, "qty"]
print(f"  net_weight_kg qty 복사: {mask_kg.sum()}개")

comp = (
    df_b
    .groupby(["hs6", "year", "country_name_kr", "eu_country_code"])["net_weight_kg"]
    .mean()
    .groupby(["hs6", "year", "country_name_kr"])
    .sum()
    .reset_index()
)

comp_pivot = comp.pivot_table(
    index=["hs6", "year"],
    columns="country_name_kr",
    values="net_weight_kg",
    aggfunc="first"
).reset_index()

name_map = {"중국": "chn", "인도": "ind", "일본": "jpn", "튀르키예": "tur", "미국": "usa"}
comp_pivot.columns = (
    ["hs6", "year"] +
    [f"comp_eu_weight_{name_map.get(c, c)}" for c in comp_pivot.columns[2:]]
)

step5 = comp_pivot
print(f"  ✅ {len(step5):,}행\n")

### STEP 6. 테이블 조립

In [ ]:
print("=" * 60)
print("STEP 6. 조립")
print("=" * 60)

hs6_master = pd.read_csv(f"{OUT_DIR}/cbam_hs6_master.csv", dtype={"hs6": str})
hs6_master["hs6"] = hs6_master["hs6"].astype(str).str.zfill(6)
hs6_master = hs6_master.rename(columns={
    "category": "cbam_category",
    "item_name_kr": "hs6_name_kr"
})

master = (
    step1
    .merge(step2,     on=["hs6", "year"], how="left")
    .merge(step3,     on="hs6",           how="left")
    .merge(step4,     on="year",          how="left")
    .merge(step5,     on=["hs6", "year"], how="left")
    .merge(hs6_master[["hs6", "cbam_category", "hs6_name_kr"]], on="hs6", how="left")
)

col_order = [
    "hs6", "year", "hs6_name_kr", "cbam_category",
    "kr_eu_export_weight_kg", "kr_eu_export_value_usd",
    "kr_world_export_weight_kg", "kr_world_export_value_usd",
    "ci_kor", "eua_price_eur", "kau_price_eur",
    "ci_chn", "ci_ind", "ci_jpn", "ci_tur", "ci_usa",
    "comp_eu_weight_chn", "comp_eu_weight_ind",
    "comp_eu_weight_jpn", "comp_eu_weight_tur", "comp_eu_weight_usa",
]
col_order = [c for c in col_order if c in master.columns]
master = master[col_order]

print(f"  조립 후: {len(master):,}행 × {len(master.columns)}컬럼")
print()

### STEP 6.5. C1 / C2 / C3 지표 산출

- **C1** = (수출중량kg ÷ 1000) × ci_kor × max(EUA − KAU, 0)  
- **C2** = 대EU 수출금액 ÷ 전세계 수출금액 × 100  
- **C3** = ci_kor − Σ(ci_경쟁국 × 경쟁국EU수출) ÷ Σ경쟁국EU수출  
  (ci NaN인 경쟁국은 분자·분모 모두 제외)

In [ ]:
print("=" * 60)
print("STEP 6.5. C1 / C2 / C3 지표 산출")
print("=" * 60)

# C1: CBAM 잠재 부담액 (EUR)
master["c1"] = (
    master["kr_eu_export_weight_kg"] / 1000
    * master["ci_kor"]
    * (master["eua_price_eur"] - master["kau_price_eur"]).clip(lower=0)
)

# C2: 대EU 수출 의존도 (%)
world_val = master["kr_world_export_value_usd"].replace(0, float("nan"))
master["c2"] = (master["kr_eu_export_value_usd"] / world_val * 100).fillna(0)

# C3: 경쟁국 탄소 격차 (tCO2/t)
ISO_LIST = ["chn", "ind", "jpn", "tur", "usa"]

def calc_c3(row):
    w_sum, wci_sum = 0.0, 0.0
    for iso in ISO_LIST:
        ci_val = row.get(f"ci_{iso}")
        w_val  = row.get(f"comp_eu_weight_{iso}", 0)
        if pd.notna(ci_val) and pd.notna(w_val) and w_val > 0:
            wci_sum += ci_val * w_val
            w_sum   += w_val
    if w_sum == 0:
        return float("nan")
    return row["ci_kor"] - wci_sum / w_sum

master["c3"] = master.apply(calc_c3, axis=1)

print(f"  C1 범위: {master["c1"].min():.2f} ~ {master["c1"].max():,.0f} EUR")
print(f"  C2 범위: {master["c2"].min():.2f} ~ {master["c2"].max():.2f} %")
print(f"  C3 NaN : {master["c3"].isna().sum()}개 (경쟁국 ci 전부 없는 품목·연도)")
print(f"  ✅ C1/C2/C3 산출 완료\n")


### STEP 7. 결측치 처리 및 저장

- `ci_kor` 없는 4개 품목(고령토·알루미나시멘트·무수암모니아·페로크롬) → drop  
- `comp_eu_weight` 결측 → 0 (해당 연도 EU 수출 없음으로 처리)

In [ ]:
print("=" * 60)
print("STEP 7. 결측치 처리")
print("=" * 60)

comp_cols = [c for c in master.columns if c.startswith("comp_eu_weight")]

before = len(master)
master = master[master["ci_kor"].notna()].copy()
print(f"  ci_kor 없는 품목 drop: {before - len(master)}행 제거")
print(f"  (고령토, 알루미나시멘트, 무수암모니아, 페로크롬)")

for col in comp_cols:
    n = master[col].isna().sum()
    master[col] = master[col].fillna(0)
    if n > 0:
        print(f"  {col} 결측 → 0: {n}개")

print()

out_path = f"{OUT_DIR}/C123_master_table.csv"
master.to_csv(out_path, index=False, encoding="utf-8-sig")

print("=" * 60)
print("✅ 완료!")
print("=" * 60)
print(f"저장: {out_path}")
print(f"행수: {len(master):,} | 컬럼수: {len(master.columns)}")
print()
print("=== 결측률 최종 확인 ===")
for col in master.columns:
    nan_cnt = master[col].isna().sum()
    if nan_cnt > 0:
        print(f"  {col}: {nan_cnt}개 ({nan_cnt/len(master)*100:.2f}%)")
print("나머지 컬럼: 결측 없음 ✅")
print()
print("=== 연도별 행수 ===")
print(master.groupby("year").size().to_string())

---
## PART 2. C1/C2/C3 스케일링 및 CRITIC 가중치 산출

**스케일링 방법**: Robust Scaler → Quantile Transformer (output_distribution='normal')  
**가중치 방법**: CRITIC (σ_j × Σ(1 − r_jk))

### STEP 8. 데이터 로드 및 결측치 처리

In [ ]:
print("STEP 8. 데이터 로드")

df = pd.read_csv(f"{OUT_DIR}/C123_master_table.csv", dtype={"hs6": str})
df["hs6"] = df["hs6"].astype(str).str.zfill(6)

print(f"  로드 완료: {len(df):,}행 × {len(df.columns)}컬럼")
print(f"  연도 범위: {df['year'].min()} ~ {df['year'].max()}")
print(f"  hs6 수: {df['hs6'].nunique()}")

features = ["c1", "c2", "c3"]
X = df[features].copy()

print("\n결측치 처리")
nan_counts = X.isna().sum()
for col, cnt in nan_counts.items():
    if cnt > 0:
        print(f"  {col}: {cnt}개 NaN → 0 대체")
    else:
        print(f"  {col}: 결측 없음")

X = X.fillna(0)

### STEP 9. 스케일링

1. **Robust Scaler**: 중앙값·IQR 기준 → C1 극값 1차 억제  
2. **Quantile Transformer** (normal): 정규분포 강제 변환 → 밀집 구간 펼침

In [ ]:
print("\nSTEP 9. 스케일링")

robust = RobustScaler()
X_robust = robust.fit_transform(X)
print("  [1] Robust Scaler 완료")
print(f"       C1 중앙값: {robust.center_[0]:.4f}  IQR: {robust.scale_[0]:.4f}")
print(f"       C2 중앙값: {robust.center_[1]:.4f}  IQR: {robust.scale_[1]:.4f}")
print(f"       C3 중앙값: {robust.center_[2]:.4f}  IQR: {robust.scale_[2]:.4f}")

qt = QuantileTransformer(
    output_distribution="normal",
    n_quantiles=min(len(X), 1000),
    random_state=42,
)
X_scaled = qt.fit_transform(X_robust)
print("  [2] Quantile Transformer (normal) 완료")

scaled_df = pd.DataFrame(
    X_scaled,
    columns=["c1_scaled", "c2_scaled", "c3_scaled"],
    index=df.index,
)

print(f"\n  스케일링 결과 범위:")
for col in ["c1_scaled", "c2_scaled", "c3_scaled"]:
    s = scaled_df[col]
    print(f"    {col}: mean={s.mean():.4f}  std={s.std():.4f}  "
          f"min={s.min():.4f}  max={s.max():.4f}")

### STEP 10. CRITIC 가중치 산출 및 저장

$$w_j \propto \sigma_j \times \sum_{k \neq j}(1 - r_{jk})$$

In [ ]:
print("\n" + "=" * 60)
print("STEP 10. CRITIC 가중치 재계산")
print("=" * 60)

stds = scaled_df.std()
corr = scaled_df.corr()

print(f"\n  [표준편차]")
for col, s in stds.items():
    print(f"    {col}: {s:.6f}")

print(f"\n  [상관행렬]")
print(corr.round(4).to_string())

cols = ["c1_scaled", "c2_scaled", "c3_scaled"]
info = {}
for j in cols:
    conflict = sum(1 - corr.loc[j, k] for k in cols if k != j)
    info[j] = stds[j] * conflict

total_info = sum(info.values())
weights = {k: v / total_info for k, v in info.items()}

label = {"c1_scaled": "C1", "c2_scaled": "C2", "c3_scaled": "C3"}

print(f"\n  [CRITIC 정보량]")
for k, v in info.items():
    print(f"    {label[k]}: {v:.6f}")

print(f"\n  *** CRITIC 가중치 결과 ***")
for k, w in weights.items():
    print(f"    {label[k]}: {w * 100:.2f}%")
print(f"    합계: {sum(weights.values()) * 100:.2f}%")

print("\nSTEP 10. 저장")
out = pd.concat([df, scaled_df], axis=1)
out_path = f"{OUT_DIR}/C123_scaled_master_table.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"  저장 경로: {out_path}")
print(f"  행수: {len(out):,}  |  컬럼수: {len(out.columns)}")
print(f"  추가 컬럼: c1_scaled, c2_scaled, c3_scaled")
print("\n" + "=" * 60)
print("완료!")
print("=" * 60)